## Step 1: Load the Data and Split them

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the data

data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/headlines_clean.csv")

# Get only the parts we need from the csv (we don't need source and source_type)
data = data[["headline", "label"]]

# Check the class distribution
print(data["label"].value_counts())

# Split the data into training and testing sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42, stratify=data["label"])

print("Training data lenght:", len(train_data))
print("Testing data lenght:", len(test_data))


In [ ]:
## Tokenization
import sys
from transformers import AutoTokenizer
from datasets import Dataset

# Fix for 'VideoReader' ImportError in Colab
sys.modules['torchvision'] = None

# Load SlovakBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("gerulata/slovakbert")

# Convert from Pandas into Huggingface datasets
train_dataset = Dataset.from_pandas(train_data.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_data.reset_index(drop=True))

# Create the function that will turn text into vectors
def tokenize(input):
  return tokenizer(
    input["headline"],
    padding="max_length", # We always make all the parts the same lenght for efficient proccessing and storing
    truncation=True, # Cut of anything after the max allowed lenght
    max_length=64
  )

# Apply tokenizer to both sets, add as new rows so we keep the original headling if needed
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Tell HuggingFace which colums the model needs
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Restore torchvision after formatting is done
del sys.modules['torchvision']

print("Example:", train_dataset[0]['input_ids'])
print("Example:", train_dataset[0]['attention_mask'])
print("Example:", train_dataset[0]['label'])


## Step 2: Load and Train SlovakBERT

In [ ]:
from transformers import AutoModelForSequenceClassification


# Load the model
model = AutoModelForSequenceClassification.from_pretrained("gerulata/slovakbert", num_labels=2) # Two labels: legit "0" and clickbait "1"




In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight


# Since I have 178 clickbait classified titles and 269 legit, we make missclasifying the minority more expensive
# Therefore not bias the model towards it
class_weights = compute_class_weight(
    "balanced",
    classes=np.array([0, 1]), # Make the whole array with the 0 and 1
    y=train_data["label"] # Get the 0 and 1 from the train dataset and have the class computer the weights
)

print(f"Class weights: legit={class_weights[0]:.3f}, clickbait={class_weights[1]:.3f}")
# Clickbait weight will be higher because there are fewer of them

In [ ]:
import torch
from transformers import Trainer, TrainingArguments

# Convert weights to a PyTorch tensor and move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels") # Extract tru target answer (0 or 1)
        outputs = model(**inputs) # Pass remaining text tokens and masks into tranformer
        logits = outputs.logits  # Raw scores before probability conversion (logits)

        # Pass the new weighted Cross-Entropy Loss
        loss_function = torch.nn.CrossEntropyLoss(weight=weights_tensor)
        loss = loss_function(logits, labels)

        # Return the correct thing when we both train and evalueate
        return (loss, outputs) if return_outputs else loss

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,            # how many times to go through the full dataset
    per_device_train_batch_size=16, # process 16 headlines at once (GPU memory limit)
    per_device_eval_batch_size=32,  # can use larger batch for eval
    learning_rate=2e-5,             # how big the adjustment steps are
    weight_decay=0.01,              # regularization - prevents overfitting
    eval_strategy="epoch",          # evaluate on test set after each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,    # keep the version that performed best on test set
    metric_for_best_model="f1",     # use F1 to pick the best model (not accuracy, because of imbalance)
    logging_steps=10,               # print loss every 10 steps
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
# The "Report Card"
# Runs after each epoch(1 whole dataset passthorugh) on the test set
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)  # pick whichever decision has the higher score
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),  # defaults to binary F1 for the positive class (clickbait)
    }

In [ ]:
import sys
# Ensure torchvision is fully removed from sys.modules to avoid the VideoReader ImportError
if 'torchvision' in sys.modules:
    del sys.modules['torchvision']

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,    # your tokenized training data from Step 2
    eval_dataset=test_dataset,      # your tokenized test data from Step 2
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Get predictions from the best model
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)  # convert logits to class predictions
true_labels = predictions.label_ids

# Print the classification report (precision, recall, F1 per class)
print(classification_report(true_labels, preds, target_names=["legit", "clickbait"]))

# Plot confusion matrix
cm = confusion_matrix(true_labels, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["legit", "clickbait"])
disp.plot(cmap="Blues")
plt.title("SlovakBERT Clickbait Classifier - Confusion Matrix")
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/Colab Notebooks/confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
print("WRONG PREDICTIONS")
print("=================")
print()

wrong_count = 0

for i in range(len(preds)):
    if preds[i] != true_labels[i]:
        wrong_count = wrong_count + 1

        if true_labels[i] == 1:
            actual = "clickbait"
        else:
            actual = "legit"

        if preds[i] == 1:
            predicted = "clickbait"
        else:
            predicted = "legit"

        headline = test_data.iloc[i]["headline"]
        print("Headline:", headline)
        print("  True:", actual, "| Predicted:", predicted)
        print()

print("Total wrong:", wrong_count, "out of", len(preds))

In [ ]:
def classify_headline(headline):
    # Tokenize the input headline the same way we did for training
    inputs = tokenizer(
        headline,
        padding="max_length",
        truncation=True,
        max_length=64,
        return_tensors="pt"  # return pytorch tensors
    )

    # Move to GPU
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # Tell PyTorch we're not training, just predicting
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    # Convert logits to probabilities using softmax
    probabilities = torch.softmax(outputs.logits, dim=1)

    # Get the predicted class and its confidence
    predicted_class = torch.argmax(probabilities).item()
    confidence = probabilities[0][predicted_class].item()

    label = "CLICKBAIT" if predicted_class == 1 else "LEGIT"
    print("Headline:", headline)
    print("Prediction:", label)
    print("Confidence:", str(round(confidence * 100, 1)) + "%")

In [ ]:
classify_headline("Slovensko zasiahli silné búrky, bez elektriny je 10 000 domácností")

In [ ]:
classify_headline("Neuveríte, čo táto žena našla vo svojom záhrade! Video vás šokuje")


In [ ]:
classify_headline("Fico oznámil nové opatrenia proti inflácii")


In [ ]:
classify_headline("Fico oznámil nové opatrenia proti inflácii: Tento krok zmení životy tisícov Slovákov")

In [ ]:
trainer.save_model("./best_model")
tokenizer.save_pretrained("./best_model")

In [ ]:
import shutil
shutil.make_archive("best_model", "zip", ".", "best_model")